In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models
from sklearn.metrics import confusion_matrix
from PIL import Image
import collections

In [2]:
# MobileNetV2 αντί για ResNet50
# classifier[1] αντί για fc, 1280 αντί για 2048
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(in_features=1280, out_features=3)

In [3]:
#DATA AUGMENTATION

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, hue=0.2, saturation=0.3),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

In [9]:
# DATASETS - ίδιο dataset με ResNet50
train_dataset = torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_MobileNetV2/train",
                                                 transform=transform_train)

test_dataset = torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_MobileNetV2/test",
                                                transform=transform_test)

In [10]:
#Train-Test Loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=32, num_workers=2, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=32, num_workers=2, shuffle=False)

In [11]:
class_counts = collections.Counter(train_dataset.targets)
for cls_idx, count in sorted(class_counts.items()):
    cls_name = train_dataset.classes[cls_idx]
    print(f"  {cls_name}: {count} pictures")

total_samples = len(train_dataset)
class_weights = torch.tensor([
    total_samples / class_counts[i]
    for i in range(len(train_dataset.classes))], dtype=torch.float)

print(f"\nClass weights: {class_weights}")

  High: 644 pictures
  Low: 806 pictures
  NoWaste: 997 pictures

Class weights: tensor([3.7997, 3.0360, 2.4544])


In [12]:
#Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
#CrossEntropyLoss με Weighted Loss
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
model = model.to(device)

In [14]:
# FROZEN LAYERS
# Στο MobileNetV2: features = convolutional layers, classifier = FC layers
# Παγώνουμε όλα εκτός από τα τελευταία features blocks + classifier
for param in model.parameters():
    param.requires_grad = False

# Ξεπαγώνουμε τα τελευταία 3 blocks του features (ανάλογο με layer3, layer4 του ResNet)
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Ξεπαγώνουμε τον classifier
for param in model.classifier.parameters():
    param.requires_grad = True

In [15]:
#Optimizer - ίδιος με ResNet50
optimizer = optim.Adam(
    list(model.features[-3:].parameters()) +
    list(model.classifier.parameters()),
    lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

In [16]:
#TRAINING LOOP
num_epoch = 10
best_accuracy = 0.0

for epoch in range(num_epoch):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epoch}, Loss: {avg_loss:.4f}')

    #Evaluation
    model.eval()
    test_loss = 0.0
    correct = 0.0
    total = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs, dim=1)
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_test_loss = test_loss / len(test_loader)
    accuracy = 100 * correct / total
    print(f'Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.2f}%\n')

    #Scheduler step
    scheduler.step(avg_test_loss)

    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), 'MNV2_WL_TK-MobileNet_V2.pth')
        print(f'New best model saved! Accuracy: {best_accuracy:.2f}%\n')

    #Confusion Matrix
    cm = confusion_matrix(all_targets, all_predictions)
    print("Confusion Matrix:")
    print(cm)
    print("-" * 40)

Epoch 1/10, Loss: 0.5351
Test Loss: 0.3465, Test Accuracy: 86.93%

New best model saved! Accuracy: 86.93%

Confusion Matrix:
[[110  36   6]
 [  8 199   9]
 [ 17   4 223]]
----------------------------------------
Epoch 2/10, Loss: 0.4118
Test Loss: 0.3786, Test Accuracy: 89.22%

New best model saved! Accuracy: 89.22%

Confusion Matrix:
[[135  14   3]
 [ 15 191  10]
 [ 24   0 220]]
----------------------------------------
Epoch 3/10, Loss: 0.3883
Test Loss: 0.3832, Test Accuracy: 88.73%

Confusion Matrix:
[[130  21   1]
 [ 12 200   4]
 [ 19  12 213]]
----------------------------------------
Epoch 4/10, Loss: 0.3751
Test Loss: 0.2823, Test Accuracy: 91.34%

New best model saved! Accuracy: 91.34%

Confusion Matrix:
[[138   7   7]
 [ 24 187   5]
 [  6   4 234]]
----------------------------------------
Epoch 5/10, Loss: 0.3768
Test Loss: 0.3178, Test Accuracy: 90.69%

Confusion Matrix:
[[138  10   4]
 [ 23 188   5]
 [ 10   5 229]]
----------------------------------------
Epoch 6/10, Loss: 0.

In [ ]:
def predict_confidence(image_path):
    model.load_state_dict(torch.load('MNV2_WL_TK-MobileNet_V2.pth'))
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image = transform_test(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, dim=1)

        prob = confidence.item()
        predicted = predicted_class.item()
        raw_confidence = int(prob * 100)

        if predicted == 0:
            confidence_score = max(85, raw_confidence)               # High  → ≥85%
        elif predicted == 1:
            confidence_score = max(65, min(84, raw_confidence))      # Low   → 65-84%
        else:
            confidence_score = min(64, raw_confidence)               # NoWaste → ≤64%

        class_names = {0: "High", 1: "Low", 2: "NoWaste"}

        print(f"Κατηγορία: {class_names[predicted]}")
        print(f"Confidence: {confidence_score}%")
        print(f"Raw Softmax: {raw_confidence}%")
        print("---")

    return class_names[predicted], confidence_score


# Δοκιμή
class_name, confidence = predict_confidence("C:/Users/panag/Desktop/test.png")
print(f"Κατηγορία: {class_name}, Confidence: {confidence}%")

SyntaxError: unterminated string literal (detected at line 2) (2581200205.py, line 2)